In [15]:
# import zipfile
# import glob

# years = [2025]
# folder = "/home/rishi/ML Projects/Air Pollution/Datasets/epa_data"

# for f in glob.glob(f"{folder}/*.zip"):
#     if any(f.endswith(f"{year}.zip") for year in years):
#         with zipfile.ZipFile(f, "r") as z:
#             z.extractall("output_folder/")


In [16]:
import pandas as pd
import os

types=['hourly_42401',
 'hourly_88101',
 'hourly_44201',
 'hourly_42602',
 'hourly_42101',
 'hourly_81102',
]

In [17]:
def separate_and_filter(df, pol):
    df["Timestamp"] = pd.to_datetime(df["Date GMT"] + " " + df["Time GMT"], format="%Y-%m-%d %H:%M")
    s=df['Parameter Name'].iloc[0]+' ' +df['Units of Measure'].iloc[0]
    year=pd.to_datetime(df["Date Local"].iloc[0]).year
    df[s]=df['Sample Measurement']
    df["Key"]=df.apply(lambda x: (x['State Code'],x['County Code'], x['Site Num']), axis=1)
    df_f=df[['Key','Timestamp', 'Latitude', 'Longitude',s]]
    output_dir = "epa_data_by_site"
    os.makedirs(output_dir, exist_ok=True)

    for site_id, group_df in df_f.groupby("Key"):
        safe_name = f"site_{site_id[0]}_{site_id[1]}_{site_id[2]}_{pol}_{year}"
        safe_name = safe_name.replace("/", "_").replace(" ", "_")
        group_df.to_csv(os.path.join(output_dir, f"{safe_name}.csv"), index=False)
    print("Saved ", pol)

In [ ]:
base = r"/home/rishi/ML Projects/Air Pollution/EPA/output_folder/"
years = [2022,2023,2024, 2025]
output_dir = "epa_data_by_site"
os.makedirs(output_dir, exist_ok=True)
os.makedirs("joined", exist_ok=True)

In [ ]:
type_to_pol = {
    'hourly_42401': 'Sulphur Dioxide',
    'hourly_88101': 'PM2.5',
    'hourly_44201': 'Ozone',
    'hourly_42602': 'Nitrogen Dioxide',
    'hourly_42101': 'Carbon Monoxide',
    'hourly_81102': 'PM10',
}

for year in years:
    for type in types:
        path = fr"{base}{type}_{year}.csv"
        if not os.path.exists(path):
            print(f"Skipping {path} (not found)")
            continue
        df = pd.read_csv(path, low_memory=False)
        separate_and_filter(df, type_to_pol[type])

In [ ]:
import glob
from collections import defaultdict

TARGET_START = '2023-01-01'
TARGET_END   = '2025-12-31 23:00'
target_index = pd.date_range(start=TARGET_START, end=TARGET_END, freq='h')

# Group per-year CSVs by (site, pollutant)
groups = defaultdict(list)
for fpath in glob.glob(os.path.join(output_dir, "*.csv")):
    fname = os.path.basename(fpath).replace(".csv", "")
    parts = fname.split("_")
    key = "_".join(parts[:-1])
    groups[key].append((int(parts[-1]), fpath))

for key, file_list in groups.items():
    file_list.sort(key=lambda x: x[0])
    dfs = [pd.read_csv(fp) for _, fp in file_list]
    combined = pd.concat(dfs, ignore_index=True)

    combined["Timestamp"] = pd.to_datetime(combined["Timestamp"])
    combined = combined.drop_duplicates(subset="Timestamp").set_index("Timestamp").sort_index()
    combined = combined.reindex(target_index)
    combined.index.name = "Timestamp"

    combined.to_csv(os.path.join("joined", f"{key}.csv"))

print(f"Saved {len(groups)} multi-year files to joined/")